# 04 — Sorguya Duyarlı Model Seçici / Router (İP-5, RQ5)

Sorgu özelliklerinden, hangi embedding modelinin bu sorguda kazanacağını tahmin eden hafif sınıflandırıcı.

**Disiplin (ihlal = sonuç geçersiz):**

- Etiketler (kazanan model) yalnızca train/val sorularının sonuçlarından üretilir.
- Test seti seçiciye hiç gösterilmez; nihai karşılaştırma bir kez, en sonda yapılır.
- En az 5 tohum; farklar eşleştirilmiş bootstrap ile raporlanır.

In [ ]:
import sys
sys.path.insert(0, '/content/RAG-proje')
%cd /content/RAG-proje

from src.config import load_config
from src.router import sorgu_ozellikleri, ozellik_matrisi, ModelSecici

cfg = load_config()
alanlar = cfg['router']['ozellikler']
print('Router özellikleri:', alanlar)
print('Örnek:', sorgu_ozellikleri('Kayıt dondurma şartları nelerdir?', 12.4, 0.81))

## Eğitim verisi nasıl üretilir?

1. Train/val soruları tüm sistemlerde çalıştırılır (notebook 02'deki akış, `bolme` filtresi train+val).
2. Her soruda en yüksek nDCG@10 alan sistem, o sorunun etiketi olur.
3. Özellikler `sorgu_ozellikleri` ile çıkarılır; BM25–dense skor farkı için iki sistemin Top-1 skorları gerekir.
4. Sınıflandırıcı eğitilir; val üzerinde erken seçim yapılır.

In [ ]:
# Eğitim iskeleti — TODO (İP-5):
# 1) train/val soru-bazında sonuçları yükle (results/.../soru-bazinda.csv üretimi train+val için tekrarlanır)
# 2) kazanan etiketi + özellikleri birleştir -> X, y
# 3) 5 tohumla eğit, val doğruluğunu kaydet:
#
# for tohum in cfg['deney']['tohumlar']:
#     secici = ModelSecici(tur=cfg['router']['model'], tohum=tohum)
#     secici.egit(X_train, y_train)
#     tahmin = secici.tahmin(X_val)
#     ...
#
# 4) Nihai test: router'ın seçtiği sistemin skorları vs en iyi tek sistem
#    -> eslestirilmis_bootstrap_fark ile karşılaştır (src/metrics.py)
print('İskelet hazır; İP-5 haftasında doldurulacak.')

## Ablasyon bağlantısı

- **A2:** Soru tipi özelliklerini çıkar → performans değişimi
- **A3:** Biçimbilimsel özellikleri çıkar → performans değişimi
- **A6:** Seçim maliyetini (ekstra çıkarım süresi) dahil et → kalite-maliyet dengesi

Router en iyi tek modeli geçemezse bu da geçerli bir bulgudur; o durumda katkı, Türkçe sorgu özelliklerinin sistematik analizi olarak raporlanır.